# Training Postmortem

This notebook analyzes the saved training artifacts from `runs/train_run2/` and focuses on whether the agent actually improved over time.

Important context:
- `episode_summaries.csv` is a per-episode CSV export produced by the training loop, which is much faster to load than parsing the raw JSONL transition log.
- The episode number resets every time training is resumed, so the same labels can appear again in later sessions.
- The true timeline in this notebook is the record order in the CSV, not the raw episode label.
- `step_analysis_data.csv` contains per-step metrics and is available for finer-grained analysis.

Primary source of truth:
- `runs/train_run2/episode_summaries.csv`
- `runs/train_run2/step_analysis_data.csv`
- `runs/train_run2/match.log` for supporting context

The notebook computes episode-level trends, rolling win rate, reward curves, board/resource proxies, regroup statistics, and a comparison between the first training session and the resumed sessions.

In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

px.defaults.template = "plotly_white"
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def find_file(filename: str, start: Path | None = None) -> Path:
    start = start or Path.cwd()
    search_roots = [start, *start.parents]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r} from {start}")



In [2]:
n = 5

In [3]:
RUN_DIR = find_file(f"runs/big_run{n}/episode_summaries.csv").parent
SUMMARY_CSV_PATH = RUN_DIR / "episode_summaries.csv"
STEP_CSV_PATH = RUN_DIR / "step_analysis_data.csv"
MATCH_LOG_PATH = RUN_DIR / "match.log"
ROLLING_WINDOW = 50
print(f"Run directory: {RUN_DIR}")
print(f"Episode summaries CSV: {SUMMARY_CSV_PATH}")
print(f"Step analysis CSV: {STEP_CSV_PATH}")
print(f"Match log: {MATCH_LOG_PATH}")

Run directory: c:\Users\valen\Desktop\code\swu\forceteki\python_rl\runs\big_run5
Episode summaries CSV: c:\Users\valen\Desktop\code\swu\forceteki\python_rl\runs\big_run5\episode_summaries.csv
Step analysis CSV: c:\Users\valen\Desktop\code\swu\forceteki\python_rl\runs\big_run5\step_analysis_data.csv
Match log: c:\Users\valen\Desktop\code\swu\forceteki\python_rl\runs\big_run5\match.log


In [4]:
raw_df = pd.read_csv(SUMMARY_CSV_PATH)

if raw_df.empty:
    raise RuntimeError(f"No episode summaries found in {SUMMARY_CSV_PATH}")

summary_df = raw_df.copy()
summary_df["record_index"] = np.arange(1, len(summary_df) + 1)
summary_df["episode_label"] = pd.to_numeric(summary_df.get("episode"), errors="coerce")
summary_df["training_order"] = summary_df["record_index"]

# Convert known numeric columns (pd.read_csv may leave some as object if "None" strings appear)
numeric_columns = [
    "steps", "agent_rewards", "opponent_rewards", "total_rewards",
    "agent_turns", "opponent_turns", "total_reward_steps",
    "valid_actions_sum", "valid_actions_count",
    "agent_valid_actions_sum", "agent_valid_actions_count",
    "agent_ready_resources_sum", "agent_credits_sum",
    "agent_board_power_sum", "agent_board_hp_sum", "agent_board_damage_sum",
    "agent_unit_count_sum", "agent_exhausted_sum",
    "agent_base_hp_sum", "agent_leader_hp_sum",
    "agent_hand_sum", "opp_hand_sum",
    "opp_board_power_sum", "opp_board_hp_sum", "opp_board_damage_sum",
    "opp_unit_count_sum", "opp_exhausted_sum",
    "opp_base_hp_sum", "opp_leader_hp_sum",
    "agent_max_valid_actions",
    "regroup_segment_count", "regroup_action_total",
    "regroup_card_action_total", "regroup_agent_action_total", "regroup_opponent_action_total",
    "agent_reward_mean", "opponent_reward_mean", "total_reward_mean",
    "avg_valid_actions", "avg_agent_valid_actions",
    "avg_agent_ready_resources", "avg_agent_credits", "avg_agent_hand", "avg_opp_hand",
    "avg_agent_base_hp", "avg_opp_base_hp",
    "avg_agent_leader_hp", "avg_opp_leader_hp",
    "avg_agent_board_power", "avg_opp_board_power",
    "avg_agent_board_hp", "avg_opp_board_hp",
    "avg_agent_board_damage", "avg_opp_board_damage",
    "avg_agent_unit_count", "avg_opp_unit_count",
    "avg_agent_exhausted", "avg_opp_exhausted",
]
for column in numeric_columns:
    if column in summary_df.columns:
        summary_df[column] = pd.to_numeric(summary_df[column], errors="coerce")

summary_df["session_index"] = (summary_df["episode_label"].fillna(-1).diff().fillna(0) < 0).cumsum() + 1
summary_df["session_episode"] = summary_df.groupby("session_index").cumcount() + 1
summary_df["first_session"] = summary_df["session_index"].min()
summary_df["is_initial_session"] = summary_df["session_index"] == summary_df["first_session"]

print(f"Loaded {len(summary_df)} episode summaries from CSV")
print(f"Episode labels: {int(summary_df['episode_label'].min())} to {int(summary_df['episode_label'].max())} across {summary_df['episode_label'].nunique()} unique labels")
print(f"Training sessions inferred from episode resets: {int(summary_df['session_index'].nunique())}")
print(f"Match log present: {MATCH_LOG_PATH.exists()}")

Loaded 800 episode summaries from CSV
Episode labels: 1 to 800 across 800 unique labels
Training sessions inferred from episode resets: 1
Match log present: True


## Load Training and Evaluation Metrics

The run did not persist a true winner field in `episode_summary` records, so this notebook derives a clear proxy outcome from the reward margin while also preserving the raw fields for future runs that do store exact outcomes.

In [5]:
raw_df = pd.read_csv(SUMMARY_CSV_PATH)

if raw_df.empty:
    raise RuntimeError(f"No episode summaries found in {SUMMARY_CSV_PATH}")

summary_df = raw_df.copy()
summary_df["record_index"] = np.arange(1, len(summary_df) + 1)
summary_df["episode_label"] = pd.to_numeric(summary_df.get("episode"), errors="coerce")
summary_df["training_order"] = summary_df["record_index"]

# Convert all known numeric columns
numeric_columns = [
    "steps", "agent_rewards", "opponent_rewards", "total_rewards",
    "agent_turns", "opponent_turns", "total_reward_steps",
    "valid_actions_sum", "valid_actions_count",
    "agent_valid_actions_sum", "agent_valid_actions_count",
    "agent_ready_resources_sum", "agent_credits_sum",
    "agent_board_power_sum", "agent_board_hp_sum", "agent_board_damage_sum",
    "agent_unit_count_sum", "agent_exhausted_sum",
    "agent_base_hp_sum", "agent_leader_hp_sum",
    "agent_hand_sum", "opp_hand_sum",
    "opp_board_power_sum", "opp_board_hp_sum", "opp_board_damage_sum",
    "opp_unit_count_sum", "opp_exhausted_sum",
    "opp_base_hp_sum", "opp_leader_hp_sum",
    "agent_max_valid_actions",
    "regroup_segment_count", "regroup_action_total",
    "regroup_card_action_total", "regroup_agent_action_total", "regroup_opponent_action_total",
    "agent_reward_mean", "opponent_reward_mean", "total_reward_mean",
    "avg_valid_actions", "avg_agent_valid_actions",
    "avg_agent_ready_resources", "avg_agent_credits", "avg_agent_hand", "avg_opp_hand",
    "avg_agent_base_hp", "avg_opp_base_hp",
    "avg_agent_leader_hp", "avg_opp_leader_hp",
    "avg_agent_board_power", "avg_opp_board_power",
    "avg_agent_board_hp", "avg_opp_board_hp",
    "avg_agent_board_damage", "avg_opp_board_damage",
    "avg_agent_unit_count", "avg_opp_unit_count",
    "avg_agent_exhausted", "avg_opp_exhausted",
]
for column in numeric_columns:
    if column in summary_df.columns:
        summary_df[column] = pd.to_numeric(summary_df[column], errors="coerce")
    else:
        summary_df[column] = np.nan

summary_df["agent_reward_margin"] = summary_df["agent_rewards"] - summary_df["opponent_rewards"]
summary_df["proxy_win"] = summary_df["agent_reward_margin"] > 0
summary_df["proxy_result"] = np.where(summary_df["proxy_win"], "proxy_win", "proxy_loss")
summary_df["winner_text"] = summary_df.get("winner").astype("string") if "winner" in summary_df.columns else pd.Series([pd.NA] * len(summary_df), index=summary_df.index)
summary_df["has_explicit_winner"] = summary_df["winner_text"].notna() & (summary_df["winner_text"].str.lower() != "none")
summary_df["outcome_label"] = np.where(summary_df["has_explicit_winner"], summary_df["winner_text"], summary_df["proxy_result"])
summary_df["outcome_source"] = np.where(summary_df["has_explicit_winner"], "explicit", "proxy_reward_margin")
summary_df["session_index"] = (summary_df["episode_label"].fillna(-1).diff().fillna(0) < 0).cumsum() + 1
summary_df["session_episode"] = summary_df.groupby("session_index").cumcount() + 1
summary_df["relative_episode"] = summary_df["record_index"]
summary_df["first_session"] = summary_df["session_index"].min()
summary_df["is_initial_session"] = summary_df["session_index"] == summary_df["first_session"]

match_log_text = MATCH_LOG_PATH.read_text(encoding="utf-8", errors="ignore") if MATCH_LOG_PATH.exists() else ""

# print(f"Loaded {len(summary_df)} episode summaries from CSV")
# print(f"Episode labels: {int(summary_df['episode_label'].min())} to {int(summary_df['episode_label'].max())} across {summary_df['episode_label'].nunique()} unique labels")
# print(f"Training sessions inferred from episode resets: {int(summary_df['session_index'].nunique())}")
# print(f"Outcome source used: {summary_df['outcome_source'].mode().iloc[0] if not summary_df.empty else 'n/a'}")
# print(f"Match log present: {bool(match_log_text)}")
# summary_df.head(3)

## Clean and Aggregate Episode-Level Metrics

This section standardizes the stored metrics, derives a file-order training index, and prepares the proxy outcome that lets us compare progress even when the logs do not include a true winner label.

In [6]:
def rolling_mean(series: pd.Series, window: int = ROLLING_WINDOW) -> pd.Series:
    return series.rolling(window=window, min_periods=max(2, window // 2)).mean()


def safe_mean(frame: pd.DataFrame, column: str) -> float:
    if column not in frame or frame[column].dropna().empty:
        return float("nan")
    return float(frame[column].mean())


def outcome_summary(frame: pd.DataFrame) -> dict[str, float]:
    if frame.empty:
        return {"n": 0, "proxy_win_rate": float("nan"), "agent_reward_mean": float("nan"), "reward_margin_mean": float("nan"), "steps_mean": float("nan")}
    margin = frame["agent_reward_margin"]
    return {
        "n": int(len(frame)),
        "proxy_win_rate": float((margin > 0).mean()),
        "agent_reward_mean": safe_mean(frame, "agent_rewards"),
        "opponent_reward_mean": safe_mean(frame, "opponent_rewards"),
        "reward_margin_mean": float(margin.mean()),
        "steps_mean": safe_mean(frame, "steps"),
    }


overall_summary = outcome_summary(summary_df)
first_session_df = summary_df[summary_df["is_initial_session"]].copy()
resumed_sessions_df = summary_df[~summary_df["is_initial_session"]].copy()
first_session_summary = outcome_summary(first_session_df)
resumed_sessions_summary = outcome_summary(resumed_sessions_df)

comparison_table = pd.DataFrame(
    [
        {"period": "initial session", **first_session_summary},
        {"period": "resumed sessions", **resumed_sessions_summary},
        {"period": "all summaries", **overall_summary},
    ]
)
comparison_table["proxy_win_rate_pct"] = comparison_table["proxy_win_rate"] * 100.0
comparison_table["reward_margin_delta_vs_initial"] = comparison_table["reward_margin_mean"] - comparison_table.loc[0, "reward_margin_mean"]
comparison_table["agent_reward_delta_vs_initial"] = comparison_table["agent_reward_mean"] - comparison_table.loc[0, "agent_reward_mean"]

best_episodes = summary_df.nlargest(5, "agent_reward_margin")[["record_index", "episode_label", "session_index", "agent_rewards", "opponent_rewards", "agent_reward_margin", "steps", "proxy_result"]]
worst_episodes = summary_df.nsmallest(5, "agent_reward_margin")[["record_index", "episode_label", "session_index", "agent_rewards", "opponent_rewards", "agent_reward_margin", "steps", "proxy_result"]]

session_summary = summary_df.groupby("session_index").agg(
    records=("record_index", "count"),
    proxy_win_rate=("proxy_win", lambda s: float(s.mean())),
    agent_reward_mean=("agent_rewards", "mean"),
    reward_margin_mean=("agent_reward_margin", "mean"),
    steps_mean=("steps", "mean"),
).reset_index()

# print("Overall summary")
# display(pd.DataFrame([overall_summary]))
# print("\nSession summary")
# display(session_summary)
# print("\nInitial vs resumed comparison")
# display(comparison_table)
# print("\nBest episodes by reward margin")
# display(best_episodes)
# print("\nWorst episodes by reward margin")
# display(worst_episodes)

## Compute Win Rate, Rolling Averages, and Stability Metrics

The main curve here uses a proxy outcome because the stored summaries do not contain a true winner label. That still gives a useful signal for whether the policy got less competent or more stable after the 50-episode checkpoint.

In [7]:
ROLLING_WINDOW = 10
summary_df = summary_df.sort_values("record_index").reset_index(drop=True)

summary_df["rolling_proxy_win_rate"] = summary_df["proxy_win"].astype(float).rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_agent_reward"] = summary_df["agent_rewards"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_reward_margin"] = summary_df["agent_reward_margin"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_steps"] = summary_df["steps"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_avg_valid_actions"] = summary_df["avg_valid_actions"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_avg_agent_valid_actions"] = summary_df["avg_agent_valid_actions"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_avg_agent_board_power"] = summary_df["avg_agent_board_power"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_avg_opp_board_power"] = summary_df["avg_opp_board_power"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_avg_agent_board_damage"] = summary_df["avg_agent_board_damage"].rolling(ROLLING_WINDOW, min_periods=1).mean()
summary_df["rolling_avg_opp_board_damage"] = summary_df["avg_opp_board_damage"].rolling(ROLLING_WINDOW, min_periods=1).mean()

initial_session_index = int(summary_df["session_index"].min())
initial_session_df = summary_df[summary_df["session_index"] == initial_session_index]
resumed_sessions_df = summary_df[summary_df["session_index"] > initial_session_index]

print(f"Proxy win rate (all summaries): {summary_df['proxy_win'].mean():.1%}")
print(f"Proxy win rate (initial session): {initial_session_df['proxy_win'].mean():.1%}")
print(f"Proxy win rate (resumed sessions): {resumed_sessions_df['proxy_win'].mean():.1%}")
print(f"Reward margin mean shift: {resumed_sessions_df['agent_reward_margin'].mean() - initial_session_df['agent_reward_margin'].mean():+.3f}")
print(f"Agent reward mean shift: {resumed_sessions_df['agent_rewards'].mean() - initial_session_df['agent_rewards'].mean():+.3f}")
print(f"Average steps shift: {resumed_sessions_df['steps'].mean() - initial_session_df['steps'].mean():+.3f}")

# display(summary_df.loc[:, ["record_index", "episode", "session_index", "proxy_win", "rolling_proxy_win_rate", "rolling_agent_reward", "rolling_reward_margin", "rolling_steps"]].tail(10))

Proxy win rate (all summaries): 38.6%
Proxy win rate (initial session): 38.6%
Proxy win rate (resumed sessions): nan%
Reward margin mean shift: +nan
Agent reward mean shift: +nan
Average steps shift: +nan


## Plot Training Curves and Win Rate Trends

These six charts are intentionally separate so each one answers a single question. The x-axis is the record order in the log, which is the real timeline because episode labels reset when training is resumed.

In [8]:
x = summary_df["training_order"]
proxy_fig = go.Figure()
proxy_fig.add_trace(go.Scatter(x=x, y=summary_df["proxy_win"].astype(float), mode="lines+markers", name="Proxy win (episode)", line=dict(color="#94a3b8", width=2), marker=dict(size=5), opacity=0.5))
proxy_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_proxy_win_rate"], mode="lines", name=f"Rolling proxy win rate ({ROLLING_WINDOW})", line=dict(color="#ef4444", width=4)))
proxy_fig.update_layout(title="Proxy Win Rate", template="plotly_white", height=520, width=1200, margin=dict(l=40, r=30, t=70, b=40), legend=dict(orientation="h"))
proxy_fig.update_xaxes(title_text="Training record order")
proxy_fig.update_yaxes(title_text="Proxy win rate")
proxy_fig.show()

In [9]:
x = summary_df["training_order"]
reward_fig = go.Figure()
reward_fig.add_trace(go.Scatter(x=x, y=summary_df["proxy_win"].astype(float), mode="lines+markers", name="Proxy win (episode)", line=dict(color="#94a3b8", width=2), marker=dict(size=5), opacity=0.5))
reward_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_proxy_win_rate"], mode="lines", name=f"Rolling proxy win rate ({ROLLING_WINDOW})", line=dict(color="#ef4444", width=4)))
reward_fig.add_trace(go.Scatter(x=x, y=summary_df["agent_rewards"], mode="lines+markers", name="Agent reward", line=dict(color="#2563eb", width=2), marker=dict(size=5), opacity=0.5))
reward_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_agent_reward"], mode="lines", name="Rolling agent reward", line=dict(color="#1d4ed8", width=4)))
reward_fig.add_trace(go.Scatter(x=x, y=summary_df["agent_reward_margin"], mode="lines+markers", name="Reward margin", line=dict(color="#f97316", width=2, dash="dot"), marker=dict(size=5), opacity=0.45))
reward_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_reward_margin"], mode="lines", name="Rolling reward margin", line=dict(color="#ea580c", width=4)))
reward_fig.update_layout(title="Outcome and Reward Trends", template="plotly_white", height=520, width=1200, margin=dict(l=40, r=30, t=70, b=40), legend=dict(orientation="h"))
reward_fig.update_xaxes(title_text="Training record order")
reward_fig.update_yaxes(title_text="Rate / reward")
reward_fig.show()

In [10]:
x = summary_df["training_order"]
length_fig = go.Figure()
length_fig.add_trace(go.Scatter(x=x, y=summary_df["steps"], mode="lines+markers", name="Episode steps", line=dict(color="#0f766e", width=2), marker=dict(size=5), opacity=0.5))
length_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_steps"], mode="lines", name="Rolling steps", line=dict(color="#115e59", width=4)))
length_fig.update_layout(title="Episode Length", template="plotly_white", height=520, width=1200, margin=dict(l=40, r=30, t=70, b=40), legend=dict(orientation="h"))
length_fig.update_xaxes(title_text="Training record order")
length_fig.update_yaxes(title_text="Steps")
length_fig.show()

In [11]:
x = summary_df["training_order"]
action_fig = go.Figure()
action_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_valid_actions"], mode="lines+markers", name="Valid actions", line=dict(color="#7c3aed", width=2), marker=dict(size=5), opacity=0.5))
action_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_avg_valid_actions"], mode="lines", name="Rolling valid actions", line=dict(color="#6d28d9", width=4)))
action_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_agent_valid_actions"], mode="lines+markers", name="Agent valid actions", line=dict(color="#a855f7", width=2, dash="dot"), marker=dict(size=5), opacity=0.5))
action_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_avg_agent_valid_actions"], mode="lines", name="Rolling agent valid actions", line=dict(color="#9333ea", width=4, dash="dot")))
action_fig.update_layout(title="Action Economy", template="plotly_white", height=520, width=1200, margin=dict(l=40, r=30, t=70, b=40), legend=dict(orientation="h"))
action_fig.update_xaxes(title_text="Training record order")
action_fig.update_yaxes(title_text="Average / rolling proxy")
action_fig.show()

In [12]:
x = summary_df["training_order"]
board_fig = go.Figure()
board_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_agent_board_power"], mode="lines+markers", name="Agent board power", line=dict(color="#0891b2", width=2), marker=dict(size=5), opacity=0.5))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_avg_agent_board_power"], mode="lines", name="Rolling agent board power", line=dict(color="#155e75", width=4)))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_opp_board_power"], mode="lines+markers", name="Opponent board power", line=dict(color="#f59e0b", width=2, dash="dot"), marker=dict(size=5), opacity=0.5))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_avg_opp_board_power"], mode="lines", name="Rolling opponent board power", line=dict(color="#b45309", width=4)))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_agent_board_damage"], mode="lines+markers", name="Agent board damage", line=dict(color="#ef4444", width=2, dash="dash"), marker=dict(size=5), opacity=0.4))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_avg_agent_board_damage"], mode="lines", name="Rolling agent board damage", line=dict(color="#b91c1c", width=4, dash="dash")))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_opp_board_damage"], mode="lines+markers", name="Opponent board damage", line=dict(color="#7c3aed", width=2, dash="dot"), marker=dict(size=5), opacity=0.4))
board_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_avg_opp_board_damage"], mode="lines", name="Rolling opponent board damage", line=dict(color="#6d28d9", width=4, dash="dot")))
board_fig.update_layout(title="Board Power and Damage Pressure", template="plotly_white", height=520, width=1200, margin=dict(l=40, r=30, t=70, b=40), legend=dict(orientation="h"))
board_fig.update_xaxes(title_text="Training record order")
board_fig.update_yaxes(title_text="Average / rolling proxy")
board_fig.show()

In [13]:
x = summary_df["training_order"]
resource_fig = go.Figure()
resource_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_agent_hand"], mode="lines+markers", name="Agent hand", line=dict(color="#14b8a6", width=2), marker=dict(size=5), opacity=0.5))
# resource_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_agent_hand"], mode="lines", name="Rolling agent hand", line=dict(color="#0f766e", width=4)))
resource_fig.add_trace(go.Scatter(x=x, y=summary_df["avg_agent_ready_resources"], mode="lines+markers", name="Ready resources", line=dict(color="#22c55e", width=2, dash="dot"), marker=dict(size=5), opacity=0.5))
# resource_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_agent_ready_resources"], mode="lines", name="Rolling ready resources", line=dict(color="#15803d", width=4)))
# resource_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_regroup_segments"], mode="lines", name="Rolling regroup segments", line=dict(color="#dc2626", width=3, dash="dash")))
# resource_fig.add_trace(go.Scatter(x=x, y=summary_df["rolling_regroup_actions"], mode="lines", name="Rolling regroup actions", line=dict(color="#991b1b", width=3, dash="dot")))
resource_fig.update_layout(title="Hand, Resources, and Regroup Pressure", template="plotly_white", height=520, width=1200, margin=dict(l=40, r=30, t=70, b=40), legend=dict(orientation="h"))
resource_fig.update_xaxes(title_text="Training record order")
resource_fig.update_yaxes(title_text="Average / rolling proxy")
resource_fig.show()

## Export Summary Tables and Figures

The last section saves the cleaned tables and Plotly figures to disk and prints a concise interpretation of the training run.

In [14]:
EXPORT_DIR = RUN_DIR / "analysis_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(EXPORT_DIR / "episode_summary_cleaned.csv", index=False)
comparison_table.to_csv(EXPORT_DIR / "session_comparison.csv", index=False)
best_episodes.to_csv(EXPORT_DIR / "best_episodes.csv", index=False)
worst_episodes.to_csv(EXPORT_DIR / "worst_episodes.csv", index=False)
session_summary.to_csv(EXPORT_DIR / "session_summary.csv", index=False)

figure_exports = {
    "proxy_win_rate": proxy_fig,
    "reward_trends": reward_fig,
    "episode_length": length_fig,
    "action_economy": action_fig,
    "board_pressure": board_fig,
    "resources_regroup": resource_fig,
    # "session_comparison": checkpoint_fig,
}

for name, figure in figure_exports.items():
    figure.write_html(EXPORT_DIR / f"{name}.html")
    try:
        figure.write_image(EXPORT_DIR / f"{name}.png", scale=2)
    except Exception:
        pass

image_status = "HTML exported; PNG exported where supported"

initial_proxy = first_session_summary["proxy_win_rate"]
resumed_proxy = resumed_sessions_summary["proxy_win_rate"]
initial_reward = first_session_summary["agent_reward_mean"]
resumed_reward = resumed_sessions_summary["agent_reward_mean"]
margin_shift = resumed_sessions_summary["reward_margin_mean"] - first_session_summary["reward_margin_mean"]
step_shift = resumed_sessions_summary["steps_mean"] - first_session_summary["steps_mean"]

interpretation_lines = [
    f"There are {len(summary_df)} episode summaries spanning {summary_df['session_index'].nunique()} resumed training sessions.",
    f"The logs do not include a true winner label for this run, so the notebook uses reward margin as a proxy outcome.",
    f"Proxy win rate is {initial_proxy:.1%} in the initial session and {resumed_proxy:.1%} in the resumed sessions.",
    f"Average agent reward moved from {initial_reward:.3f} to {resumed_reward:.3f}, while the mean reward margin changed by {margin_shift:+.3f}.",
    f"Average episode length dropped by {step_shift:+.1f} steps, which is consistent with a more passive policy or earlier game endings in later sessions.",
    "The later sessions do not show a clear improvement signal; the proxy win rate is lower and the reward profile is weaker than in the initial run.",
]

print("Export status:", image_status)
print("\nInterpretation:")
for line in interpretation_lines:
    print(f"- {line}")

print("\nExported files:")
for path in sorted(EXPORT_DIR.iterdir()):
    print(f"- {path.name}")

Export status: HTML exported; PNG exported where supported

Interpretation:
- There are 800 episode summaries spanning 1 resumed training sessions.
- The logs do not include a true winner label for this run, so the notebook uses reward margin as a proxy outcome.
- Proxy win rate is 38.6% in the initial session and nan% in the resumed sessions.
- Average agent reward moved from -6.334 to nan, while the mean reward margin changed by +nan.
- Average episode length dropped by +nan steps, which is consistent with a more passive policy or earlier game endings in later sessions.
- The later sessions do not show a clear improvement signal; the proxy win rate is lower and the reward profile is weaker than in the initial run.

Exported files:
- action_economy.html
- action_economy.png
- best_episodes.csv
- board_pressure.html
- board_pressure.png
- episode_length.html
- episode_length.png
- episode_summary_cleaned.csv
- proxy_win_rate.html
- proxy_win_rate.png
- resources_regroup.html
- resource

In [15]:
# --- Turn-level analysis setup ---
TURN_CSV_PATH = RUN_DIR / "turn_summaries.csv"

turn_raw = pd.read_csv(TURN_CSV_PATH)

if turn_raw.empty:
    raise RuntimeError(f"No turn summaries found in {TURN_CSV_PATH}")

turn_df = turn_raw.copy()
turn_df["episode"] = pd.to_numeric(turn_df["episode"], errors="coerce")
turn_df["turn_number"] = pd.to_numeric(turn_df["turn_number"], errors="coerce")
turn_df["record_index"] = np.arange(1, len(turn_df) + 1)

# Build a training-order column that maps episode → record_order from the summary_df
episode_order = summary_df[["episode_label", "training_order"]].dropna().drop_duplicates("episode_label")
episode_order["episode_label"] = episode_order["episode_label"].astype(int)
turn_df = turn_df.merge(
    episode_order.rename(columns={"episode_label": "episode"}),
    on="episode", how="left"
)
turn_df["training_order"] = turn_df["training_order"].fillna(turn_df["record_index"])

# Convert all stat columns to numeric
stat_cols = [c for c in turn_df.columns if c not in {"run_id", "episode", "turn_number", "step_index", "record_index", "training_order"}]
for c in stat_cols:
    turn_df[c] = pd.to_numeric(turn_df[c], errors="coerce")

# Optional: drop turn 0 if it's just the setup phase
if turn_df["turn_number"].min() == 0:
    turn_df = turn_df[turn_df["turn_number"] > 0]

print(f"Loaded {len(turn_df)} turn summaries across "
      f"{turn_df['episode'].nunique()} episodes, "
      f"{turn_df['turn_number'].nunique()} unique turn numbers")
print(f"Turn numbers present: {sorted(turn_df['turn_number'].unique())}")
turn_df.head(3)

Loaded 13987 turn summaries across 800 episodes, 24 unique turn numbers
Turn numbers present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


,run_id,episode,turn_number,step_index,agent_base_hp,agent_leader_hp,agent_board_power,agent_board_hp,agent_board_damage,agent_unit_count,agent_exhausted_count,agent_ready_resources,agent_credits,agent_hand_count,opp_base_hp,opp_leader_hp,opp_board_power,opp_board_hp,opp_board_damage,opp_unit_count,opp_exhausted_count,opp_hand_count,record_index,training_order
0,20260718T213208Z-87d368b2,1,1,7,34.0,0.0,0.0,34.0,0.0,0.0,0.0,2.0,0.0,6.0,28.0,0.0,0.0,28.0,0.0,0.0,0.0,6.0,1,1
1,20260718T213208Z-87d368b2,1,1,8,34.0,0.0,0.0,34.0,0.0,0.0,0.0,2.0,0.0,6.0,28.0,0.0,0.0,28.0,0.0,0.0,0.0,6.0,2,1
2,20260718T213208Z-87d368b2,1,2,11,34.0,0.0,0.0,34.0,0.0,0.0,0.0,2.0,0.0,8.0,28.0,0.0,0.0,28.0,0.0,0.0,0.0,8.0,3,1


In [16]:
# --- Per-turn evolution across training ---
TURN = 6  # <<< change this to the turn you want to inspect

turn_subset = turn_df[turn_df["turn_number"] == TURN].sort_values("training_order")

if turn_subset.empty:
    raise RuntimeError(f"No data for turn {TURN}. Available turns: {sorted(turn_df['turn_number'].unique())}")

x = turn_subset["training_order"]

# Each tuple is: (subplot_title, [(col, label, color), (col, label, color)])
subplot_groups = [
    ("Board Power", [
        ("agent_board_power", "Agent", "#0891b2"),
        ("opp_board_power", "Opponent", "#f59e0b"),
    ]),
    ("Board HP", [
        ("agent_board_hp", "Agent", "#06b6d4"),
        ("opp_board_hp", "Opponent", "#d97706"),
    ]),
    ("Units Deployed", [
        ("agent_unit_count", "Agent", "#22c55e"),
        ("opp_unit_count", "Opponent", "#ef4444"),
    ]),
    ("Hand Size", [
        ("agent_hand_count", "Agent", "#a855f7"),
        ("opp_hand_count", "Opponent", "#ec4899"),
    ]),
    ("Resources & Credits", [
        ("agent_ready_resources", "Ready", "#14b8a6"),
        ("agent_credits", "Credits", "#f97316"),
    ]),
    ("Base HP", [
        ("agent_base_hp", "Agent", "#3b82f6"),
        ("opp_base_hp", "Opponent", "#dc2626"),
    ]),
]

fig = make_subplots(rows=3, cols=2,
                    subplot_titles=[g[0] for g in subplot_groups],
                    vertical_spacing=0.12)

for idx, (_, metrics) in enumerate(subplot_groups):
    row = idx // 2 + 1
    col_pos = idx % 2 + 1
    for col, label, color in metrics:
        if col not in turn_subset.columns:
            continue
        fig.add_trace(
            go.Scatter(x=x, y=turn_subset[col], mode="lines+markers",
                       name=label, line=dict(color=color, width=2),
                       marker=dict(size=5), opacity=0.7),
            row=row, col=col_pos
        )
        fig.add_trace(
            go.Scatter(x=x, y=turn_subset[col].rolling(10, min_periods=1).mean(),
                       mode="lines", name=f"{label} (avg10)",
                       line=dict(color=color, width=3, dash="dash")),
            row=row, col=col_pos
        )

fig.update_layout(
    height=1000, width=1400,
    title_text=f"Turn {TURN} Stats Across Training Episodes",
    template="plotly_white",
    showlegend=False,
    margin=dict(l=40, r=30, t=80, b=40)
)
fig.update_xaxes(title_text="Training order", row=3, col=1)
fig.update_xaxes(title_text="Training order", row=3, col=2)
fig.show()

# --- Quick summary table ---
all_metric_cols = [c for _, metrics in subplot_groups for c, _, _ in metrics if c in turn_subset.columns]
n_late = max(5, len(turn_subset) // 4)
early_mean = turn_subset.head(n_late)[all_metric_cols].mean()
late_mean  = turn_subset.tail(n_late)[all_metric_cols].mean()
delta = late_mean - early_mean

summary_table = pd.DataFrame({"early_mean": early_mean, "late_mean": late_mean, "delta": delta}).round(2)
# display(summary_table)

In [17]:
# --- 1. Units on board vs exhausted units (for a chosen turn) ---
TURN_EX = 6  # <<< change to inspect a different turn

turn_ex = turn_df[turn_df["turn_number"] == TURN_EX].sort_values("training_order")

if turn_ex.empty:
    raise RuntimeError(f"No data for turn {TURN_EX}")

fig_ex = go.Figure()
fig_ex.add_trace(go.Scatter(
    x=turn_ex["training_order"], y=turn_ex["agent_unit_count"],
    mode="lines+markers", name="Agent total units",
    line=dict(color="#22c55e", width=2), marker=dict(size=5), opacity=0.6
))
fig_ex.add_trace(go.Scatter(
    x=turn_ex["training_order"], y=turn_ex["agent_exhausted_count"],
    mode="lines+markers", name="Agent exhausted units",
    line=dict(color="#ef4444", width=2), marker=dict(size=5), opacity=0.6
))
fig_ex.add_trace(go.Scatter(
    x=turn_ex["training_order"],
    y=turn_ex["agent_unit_count"] - turn_ex["agent_exhausted_count"],
    mode="lines+markers", name="Agent ready (unexhausted) units",
    line=dict(color="#f97316", width=2, dash="dot"), marker=dict(size=5), opacity=0.6
))
fig_ex.add_trace(go.Scatter(
    x=turn_ex["training_order"],
    y=turn_ex["agent_unit_count"].rolling(10, min_periods=1).mean(),
    mode="lines", name="Agent total units (avg10)",
    line=dict(color="#15803d", width=4)
))
fig_ex.add_trace(go.Scatter(
    x=turn_ex["training_order"],
    y=turn_ex["agent_exhausted_count"].rolling(10, min_periods=1).mean(),
    mode="lines", name="Agent exhausted (avg10)",
    line=dict(color="#b91c1c", width=4)
))

fig_ex.update_layout(
    title=f"Units on Board vs Exhausted — Turn {TURN_EX} Across Training",
    xaxis_title="Training order", yaxis_title="Count",
    height=500, width=1200, template="plotly_white",
    legend=dict(orientation="h", y=1.12)
)
fig_ex.show()

In [21]:
# --- 2. Resources at each turn for a chosen episode ---
EPISODE = 751  # <<< change to the episode number you want to inspect

ep_turns = turn_df[turn_df["episode"] == EPISODE].sort_values("turn_number")

if ep_turns.empty:
    raise RuntimeError(f"No turn data for episode {EPISODE}")

# Deduplicate: keep only the last row per turn (the regroup-phase snapshot)
ep_turns = ep_turns.groupby("turn_number", as_index=False).last()

fig_res = go.Figure()
fig_res.add_trace(go.Scatter(
    x=ep_turns["turn_number"], y=ep_turns["agent_ready_resources"],
    mode="lines+markers+text", name="Ready resources",
    line=dict(color="#14b8a6", width=3), marker=dict(size=10),
    text=ep_turns["agent_ready_resources"].round(1),
    textposition="top center"
))
fig_res.add_trace(go.Scatter(
    x=ep_turns["turn_number"], y=ep_turns["agent_credits"],
    mode="lines+markers+text", name="Credits",
    line=dict(color="#f97316", width=3), marker=dict(size=10),
    text=ep_turns["agent_credits"].round(1),
    textposition="bottom center"
))

fig_res.update_layout(
    title=f"End-of-Turn Resources — Episode {int(EPISODE)}",
    xaxis_title="Turn number", yaxis_title="Count",
    height=450, width=1000, template="plotly_white",
    legend=dict(orientation="h", y=1.12)
)
fig_res.update_xaxes(dtick=1)
fig_res.show()

# Also show a quick table
display(ep_turns[["turn_number", "agent_ready_resources", "agent_credits",
                   "agent_hand_count", "agent_unit_count"]].round(1))

,turn_number,agent_ready_resources,agent_credits,agent_hand_count,agent_unit_count
0,1,0.0,0.0,5.0,1.0
1,2,1.0,0.0,5.0,2.0
2,3,2.0,0.0,5.0,2.0
3,4,1.0,0.0,5.0,1.0
4,5,2.0,0.0,5.0,1.0
5,6,0.0,0.0,4.0,1.0
6,7,1.0,0.0,4.0,0.0
7,8,2.0,0.0,3.0,0.0
8,9,5.0,0.0,3.0,1.0


In [19]:
# --- 3. Last turn of each episode across training ---
# Keep only the highest turn_number per episode (the last recorded turn snapshot)
last_turn_idx = turn_df.groupby("episode")["turn_number"].idxmax()
last_turn_df = turn_df.loc[last_turn_idx].sort_values("training_order").copy()

# Deduplicate within that last turn (keep the regroup-phase row, i.e. last step_index)
last_turn_df = last_turn_df.groupby("episode", as_index=False).last().sort_values("training_order")

print(f"Found {len(last_turn_df)} episodes with last-turn data")
print(f"Avg last turn number: {last_turn_df['turn_number'].mean():.1f} "
      f"(min={int(last_turn_df['turn_number'].min())}, max={int(last_turn_df['turn_number'].max())})")

x_lt = last_turn_df["training_order"]

fig_lt = make_subplots(rows=3, cols=2,
    subplot_titles=("Board Power (last turn)", "Board HP (last turn)",
                    "Units (last turn)", "Exhausted (last turn)",
                    "Resources & Hand (last turn)", "Base HP (last turn)"),
    vertical_spacing=0.12)

lt_plots = [
    # (row, col, [(col_name, label, color), ...])
    (1, 1, [
        ("agent_board_power", "Agent power", "#0891b2"),
        ("opp_board_power", "Opponent power", "#f59e0b"),
    ]),
    (1, 2, [
        ("agent_board_hp", "Agent HP", "#06b6d4"),
        ("opp_board_hp", "Opponent HP", "#d97706"),
    ]),
    (2, 1, [
        ("agent_unit_count", "Agent units", "#22c55e"),
        ("opp_unit_count", "Opponent units", "#ef4444"),
    ]),
    (2, 2, [
        ("agent_exhausted_count", "Agent exhausted", "#dc2626"),
        ("agent_unit_count", "Agent total units (ref)", "#22c55e"),
    ]),
    (3, 1, [
        ("agent_ready_resources", "Ready resources", "#14b8a6"),
        ("agent_hand_count", "Hand size", "#a855f7"),
    ]),
    (3, 2, [
        ("agent_base_hp", "Agent base HP", "#3b82f6"),
        ("opp_base_hp", "Opponent base HP", "#dc2626"),
    ]),
]

for r, c, metrics_list in lt_plots:
    for col_name, label, color in metrics_list:
        if col_name not in last_turn_df.columns:
            continue
        fig_lt.add_trace(
            go.Scatter(x=x_lt, y=last_turn_df[col_name],
                       mode="lines+markers", name=label,
                       line=dict(color=color, width=2),
                       marker=dict(size=5), opacity=0.6),
            row=r, col=c
        )
        fig_lt.add_trace(
            go.Scatter(x=x_lt,
                       y=last_turn_df[col_name].rolling(10, min_periods=1).mean(),
                       mode="lines", name=f"{label} (avg10)",
                       line=dict(color=color, width=3, dash="dash")),
            row=r, col=c
        )

# Add steps-per-episode trend as a bonus
fig_steps = go.Figure()
fig_steps.add_trace(go.Scatter(
    x=x_lt, y=last_turn_df["turn_number"],
    mode="lines+markers", name="Last turn number (= game length in turns)",
    line=dict(color="#7c3aed", width=2), marker=dict(size=5), opacity=0.6
))
fig_steps.add_trace(go.Scatter(
    x=x_lt,
    y=last_turn_df["turn_number"].rolling(10, min_periods=1).mean(),
    mode="lines", name="Game length (avg10)",
    line=dict(color="#6d28d9", width=4)
))
fig_steps.update_layout(
    title="Game Length (Last Turn Number) Across Training",
    xaxis_title="Training order", yaxis_title="Last turn number",
    height=400, width=1200, template="plotly_white",
    legend=dict(orientation="h", y=1.12)
)
fig_steps.show()

fig_lt.update_layout(
    height=1000, width=1400,
    title_text="Last Turn Metrics Across Training",
    template="plotly_white", showlegend=False,
    margin=dict(l=40, r=30, t=80, b=40)
)
fig_lt.update_xaxes(title_text="Training order", row=3, col=1)
fig_lt.update_xaxes(title_text="Training order", row=3, col=2)
fig_lt.show()

# Quick comparison
n_early = max(5, len(last_turn_df) // 4)
early_lt = last_turn_df.head(n_early).mean(numeric_only=True)
late_lt = last_turn_df.tail(n_early).mean(numeric_only=True)
delta_lt = late_lt - early_lt
comp_cols = ["turn_number", "agent_board_power", "opp_board_power",
             "agent_board_hp", "opp_board_hp",
             "agent_unit_count", "opp_unit_count",
             "agent_exhausted_count",
             "agent_ready_resources", "agent_hand_count",
             "agent_base_hp", "opp_base_hp"]
comp_lt = pd.DataFrame({
    "early": early_lt[comp_cols], "late": late_lt[comp_cols], "delta": delta_lt[comp_cols]
}).round(2)
# display(comp_lt)

Found 800 episodes with last-turn data
Avg last turn number: 8.6 (min=3, max=24)
